In [1]:
import os,glob,warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

traj_dir=r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Mentor submissions\Pharma datasets\Process"
meta_dir=r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Mentor submissions\Pharma datasets"
lab_path=os.path.join(meta_dir,"Laboratory.csv")

def read_csv(path):
    sep=";" if open(path,errors="ignore").readline().count(";")>0 else ","
    return pd.read_csv(path,sep=sep)

traj_files=sorted([f for f in glob.glob(os.path.join(traj_dir,"*.csv")) if os.path.basename(f)[:-4].isdigit()],key=lambda x:int(os.path.basename(x)[:-4]))
print(len(traj_files),"trajectory files")

25 trajectory files


In [2]:
lab=read_csv(lab_path)
lab_batches=set(lab["batch"].unique())
traj_batches=set()
for f in traj_files:
    traj_batches|=set(read_csv(f)["batch"].unique())
print("Batches in Laboratory.csv:",len(lab_batches))
print("Batches in trajectory files:",len(traj_batches))
print("Orphans (trajectory but no lab):",len(traj_batches-lab_batches))
print("Orphans (lab but no trajectory):",len(lab_batches-traj_batches))

Batches in Laboratory.csv: 1005
Batches in trajectory files: 1005
Orphans (trajectory but no lab): 0
Orphans (lab but no trajectory): 0


In [3]:
batch_codes={}
for f in traj_files:
    cid=int(os.path.basename(f)[:-4])
    for b in read_csv(f)["batch"].unique():
        batch_codes.setdefault(b,[]).append(cid)
dups={b:c for b,c in batch_codes.items() if len(c)>1}
print("Batch IDs in more than one product file:",len(dups))

Batch IDs in more than one product file: 0


In [4]:
total=0
per_file=[]
for f in traj_files:
    df=read_csv(f)
    d=df.duplicated().sum()
    total+=d
    per_file.append((int(os.path.basename(f)[:-4]),len(df),d))
print("Identical-neighbour rows total:",f"{total:,}",f"({total/sum(p[1] for p in per_file)*100:.1f}%)")
for c,n,d in per_file:
    if d>0:
        print(f"  code {c}: {d:,} of {n:,} rows")

Identical-neighbour rows total: 781,425 (16.6%)
  code 1: 20,920 of 106,878 rows
  code 13: 228,835 of 971,164 rows
  code 15: 155,170 of 493,017 rows
  code 16: 360 of 55,171 rows
  code 17: 226,108 of 843,959 rows
  code 23: 150,032 of 694,893 rows


In [5]:
df=read_csv([f for f in traj_files if os.path.basename(f)[:-4]=="13"][0])
dup_rows=df[df.duplicated(keep=False)]
key=["tbl_speed","main_comp","ejection","stiffness"]
frac_zero=(dup_rows[key].sum(axis=1)==0).mean()*100
print("Duplicated rows in code 13:",f"{len(dup_rows):,}")
print(f"Fraction of those rows where all key sensors are zero: {frac_zero:.0f}%")
print(df[df.duplicated(keep=False)].head(6).to_string())

Duplicated rows in code 13: 277,066
Fraction of those rows where all key sensors are zero: 4%
        timestamp  campaign  batch  code  tbl_speed  fom  main_comp  tbl_fill  SREL  pre_comp  produced  waste  cyl_main  cyl_pre  stiffness  ejection
1  07052019 20:15        48    188    13        0.0  0.0        0.0      6.15   0.0       0.0       0.0    0.0      2.15      5.0        0.0       0.0
2  07052019 20:15        48    188    13        0.0  0.0        0.0      6.15   0.0       0.0       0.0    0.0      2.15      5.0        0.0       0.0
3  07052019 20:15        48    188    13        0.0  0.0        0.0      6.15   0.0       0.0       0.0    0.0      2.15      5.0        0.0       0.0
4  07052019 20:15        48    188    13        0.0  0.0        0.0      6.15   0.0       0.0       0.0    0.0      2.15      5.0        0.0       0.0
5  07052019 20:15        48    188    13        0.0  0.0        0.0      6.15   0.0       0.0       0.0    0.0      2.15      5.0        0.0       0.0


In [6]:
df=read_csv([f for f in traj_files if os.path.basename(f)[:-4]=="13"][0])
print("Sample timestamp values:")
print(df["timestamp"].head(3).tolist())
ts=pd.to_datetime(df["timestamp"],errors="coerce",dayfirst=True)
print("Parsed ok:",ts.notna().mean()*100,"%")
b=df[df["batch"]==df["batch"].iloc[0]]
print("Rows in first batch:",len(b))
print("Unique timestamps in first batch:",b["timestamp"].nunique())
print("Avg rows per timestamp:",round(len(b)/b["timestamp"].nunique(),1))

Sample timestamp values:
['07052019 20:14', '07052019 20:15', '07052019 20:15']
Parsed ok: 0.0 %
Rows in first batch: 6082
Unique timestamps in first batch: 1015
Avg rows per timestamp: 6.0


In [7]:
df=read_csv([f for f in traj_files if os.path.basename(f)[:-4]=="13"][0])
ts=pd.to_datetime(df["timestamp"],format="%d%m%Y %H:%M",errors="coerce")
print("Code 13 parsed ok:",round(ts.notna().mean()*100,1),"%")

df1=read_csv([f for f in traj_files if os.path.basename(f)[:-4]=="22"][0])
print("Code 22 sample:",df1["timestamp"].head(2).tolist())
ts2=pd.to_datetime(df1["timestamp"],format="%Y-%m-%d %H:%M:%S",errors="coerce")
print("Code 22 with seconds-format parsed ok:",round(ts2.notna().mean()*100,1),"%")

Code 13 parsed ok: 100.0 %
Code 22 sample: ['2018-12-10 15:10:54', '2018-12-10 15:11:04']
Code 22 with seconds-format parsed ok: 100.0 %


In [8]:
sensors=["tbl_speed","main_comp","pre_comp","ejection","tbl_fill","stiffness"]
neg=False
for f in traj_files:
    df=read_csv(f)
    for s in sensors:
        if s in df.columns and (df[s]<0).any():
            print(f"code {os.path.basename(f)[:-4]} {s}: {(df[s]<0).sum()} negatives")
            neg=True
if not neg:
    print("No negative values in force/speed/fill sensors. Pass.")

No negative values in force/speed/fill sensors. Pass.


In [9]:
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
print("Lab rows:",len(lab),"| unique batches:",lab["batch"].nunique())
print("Duplicate batch rows in lab:",lab.duplicated(subset=["batch"]).sum())
print("Missing target values:",lab[targets].isna().sum().sum())
print()
for t in targets:
    s=pd.to_numeric(lab[t],errors="coerce")
    q1,q3=s.quantile(0.25),s.quantile(0.75)
    iqr=q3-q1
    n=((s<q1-1.5*iqr)|(s>q3+1.5*iqr)).sum()
    print(f"{t}: {n} outliers ({n/len(s)*100:.1f}%), range [{s.min():.2f}, {s.max():.2f}]")

Lab rows: 1005 | unique batches: 1005
Duplicate batch rows in lab: 0
Missing target values: 0

dissolution_av: 12 outliers (1.2%), range [82.50, 102.67]
tbl_av_hardness: 116 outliers (11.5%), range [27.00, 102.00]
tbl_rsd_weight: 67 outliers (6.7%), range [0.41, 9.90]
fct_tensile: 2 outliers (0.2%), range [1.04, 3.04]


Leakage check the most important.

In [10]:
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
print("Variance in each target explained by product code alone:")
for t in targets:
    s=pd.to_numeric(lab[t],errors="coerce")
    gm=s.mean()
    ss_total=((s-gm)**2).sum()
    ss_between=lab.groupby("code").apply(lambda g:len(g)*(pd.to_numeric(g[t],errors="coerce").mean()-gm)**2).sum()
    print(f"  {t}: {ss_between/ss_total*100:.0f}%")

Variance in each target explained by product code alone:
  dissolution_av: 40%
  tbl_av_hardness: 89%
  tbl_rsd_weight: 7%
  fct_tensile: 86%
